In [ ]:
# Notebook 02: RDKit descriptors & fingerprints

This notebook **inspects** cached OpenADNet descriptors (`src/features_data.py`), runs **PCA**, and measures **feature importance** with impurity-based scores, **permutation importance**, and **SHAP** (low-dimensional physicochemical descriptors; fingerprint models use PCA space or top bit indices for tractability).

In [ ]:
# Repo root: works if cwd is repo root or notebooks/
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from rdkit import Chem

_REPO = Path.cwd().resolve()
if not (_REPO / "src" / "features_data.py").exists():
    _REPO = _REPO.parent
sys.path.insert(0, str(_REPO / "src"))

from features_data import RDKIT_PHYS_PROP_NAMES, build_descriptor_matrix

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# PXR train split (same source as notebooks/01.ipynb)
train = pd.read_csv(
    "hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv"
)

# Subsample for faster SHAP / plots (remove or raise N_MAX for full train)
N_MAX = 800
if len(train) > N_MAX:
    train = train.sample(n=N_MAX, random_state=42).reset_index(drop=True)

TARGET = "pEC50"
y = train[TARGET].to_numpy(dtype=np.float64)
valid_y = np.isfinite(y)

mols = [Chem.MolFromSmiles(s) for s in train["SMILES"]]
ok_mol = np.array([m is not None for m in mols], dtype=bool)
mask = valid_y & ok_mol
train_f = train.loc[mask].reset_index(drop=True)
mols_f = [mols[i] for i, k in enumerate(mask) if k]
y = y[mask]
print(len(train_f), "rows with finite target and parseable SMILES")

In [ ]:
FP_NAME = "morgan_r2_bits_2048"  # try e.g. morgan_r1_count_1024, rdkit_bits_2048

X_phys = build_descriptor_matrix("rdkit_phys_props", mols_f)
X_fp = build_descriptor_matrix(FP_NAME, mols_f)

phys_finite = np.isfinite(X_phys).all(axis=1)
X_phys = X_phys[phys_finite]
X_fp = X_fp[phys_finite]
y = y[phys_finite]
train_f = train_f.loc[phys_finite].reset_index(drop=True)
mols_f = [m for m, k in zip(mols_f, phys_finite) if k]

phys_df = pd.DataFrame(X_phys, columns=list(RDKIT_PHYS_PROP_NAMES))
print("Physicochemical descriptors (first rows):")
display(phys_df.head())
print(f"\nFingerprint `{FP_NAME}` shape:", X_fp.shape, "dtype:", X_fp.dtype)
if np.nanmax(X_fp) <= 1.0:
    print("Mean bit density (fraction of 1s):", float(np.mean(X_fp)))
else:
    print("Count fingerprint — mean entry / max:", float(np.mean(X_fp)), float(np.nanmax(X_fp)))

In [ ]:
# Quick views: descriptor correlations & fingerprint column activity
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
c = phys_df.corr(numeric_only=True)
sns.heatmap(c, ax=axes[0], cmap="vlag", center=0, square=True)
axes[0].set_title("RDKit phys props — correlation")

col_mean = np.mean(X_fp, axis=0)
axes[1].hist(col_mean, bins=40, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Fraction of molecules with bit ON (per column)")
axes[1].set_ylabel("Count of bits")
axes[1].set_title(f"Per-bit prevalence — {FP_NAME}")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sc_p = StandardScaler()
Xp = sc_p.fit_transform(X_phys)

pca_phys = PCA(n_components=min(8, Xp.shape[1]))
Zp = pca_phys.fit_transform(Xp)

sc_f = StandardScaler()
Xf = sc_f.fit_transform(X_fp)
pca_fp = PCA(n_components=min(40, Xf.shape[0] - 1, Xf.shape[1]))
Zf = pca_fp.fit_transform(Xf)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].scatter(Zp[:, 0], Zp[:, 1], c=y, cmap="viridis", s=12, alpha=0.75)
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title("Phys descriptors — PCA (scaled)")

axes[1].scatter(Zf[:, 0], Zf[:, 1], c=y, cmap="viridis", s=12, alpha=0.75)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title(f"{FP_NAME} — PCA (scaled)")

ev = pca_fp.explained_variance_ratio_
axes[2].bar(range(1, len(ev) + 1), np.cumsum(ev))
axes[2].set_xlabel("Number of components"); axes[2].set_ylabel("Cumulative explained variance")
axes[2].set_title("Fingerprint PCA — cumulative variance")
axes[2].set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

print(
    f"Phys: first 3 PCs explain {pca_phys.explained_variance_ratio_[:3].sum():.1%} of variance"
)
print(
    f"FP: first 10 PCs explain {pca_fp.explained_variance_ratio_[:10].sum():.1%} of variance"
)

## Supervised models, permutation importance, and SHAP

**Physicochemical descriptors** have human-readable names, so we use a **random forest** with **permutation importance** and **SHAP TreeExplainer**.

**Fingerprints** are high-dimensional; SHAP on every bit is expensive. Here we train a forest on the **first 25 scaled PCA components** of the fingerprint matrix and explain predictions in **PC space** (inspect `pca_fp.components_` for loadings onto original bits if you need structure-level interpretation).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
import shap

N_PC_EXPLAIN = 25

# One split so phys and fingerprint models see the same train/test molecules
_idx = np.arange(len(y))
idx_train, idx_test = train_test_split(_idx, test_size=0.2, random_state=42)
X_train, X_test = X_phys[idx_train], X_phys[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
Z_fp_train = Zf[idx_train, :N_PC_EXPLAIN]
Z_fp_test = Zf[idx_test, :N_PC_EXPLAIN]
y_tr, y_te = y_train, y_test

rf_phys = RandomForestRegressor(
    n_estimators=300, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf_phys.fit(X_train, y_train)
pred = rf_phys.predict(X_test)
print(
    "Phys RF — MAE:",
    mean_absolute_error(y_test, pred),
    " R2:",
    r2_score(y_test, pred),
)

perm_phys = permutation_importance(
    rf_phys,
    X_test,
    y_test,
    n_repeats=15,
    random_state=42,
    n_jobs=-1,
)
imp_order = np.argsort(perm_phys.importances_mean)
names = list(RDKIT_PHYS_PROP_NAMES)
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh([names[i] for i in imp_order], perm_phys.importances_mean[imp_order], xerr=perm_phys.importances_std[imp_order])
ax.set_xlabel("Mean Δ R² when feature permuted (test set)")
ax.set_title("Phys descriptors — permutation importance")
plt.tight_layout()
plt.show()

rf_fp_pc = RandomForestRegressor(
    n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf_fp_pc.fit(Z_fp_train, y_tr)
pred_z = rf_fp_pc.predict(Z_fp_test)
print(
    f"FP PCA({N_PC_EXPLAIN}) RF — MAE:",
    mean_absolute_error(y_te, pred_z),
    " R2:",
    r2_score(y_te, pred_z),
)

perm_z = permutation_importance(
    rf_fp_pc,
    Z_fp_test,
    y_te,
    n_repeats=8,
    random_state=0,
    n_jobs=-1,
)
pc_labels = [f"PC{i+1}" for i in range(N_PC_EXPLAIN)]
zo = np.argsort(perm_z.importances_mean)
fig, ax = plt.subplots(figsize=(6, 6))
ax.barh([pc_labels[i] for i in zo], perm_z.importances_mean[zo], xerr=perm_z.importances_std[zo])
ax.set_xlabel("Mean Δ R² when PC permuted (test set)")
ax.set_title(f"Fingerprint PCA — permutation importance ({FP_NAME})")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP: physicochemical (small feature set — full summary plot)
shap_sample = min(400, len(X_test))
X_shap = X_test[:shap_sample]
explainer_phys = shap.TreeExplainer(rf_phys)
sv_phys = explainer_phys.shap_values(X_shap)

shap.summary_plot(
    sv_phys,
    X_shap,
    feature_names=list(RDKIT_PHYS_PROP_NAMES),
    show=False,
)
plt.title("SHAP — RDKit physicochemical descriptors")
plt.tight_layout()
plt.show()

# SHAP: forest on PCA scores (PC space)
Z_shap = Z_fp_test[: min(300, len(Z_fp_test))]
explainer_z = shap.TreeExplainer(rf_fp_pc)
sv_z = explainer_z.shap_values(Z_shap)
shap.summary_plot(
    sv_z,
    Z_shap,
    feature_names=pc_labels,
    show=False,
)
plt.title(f"SHAP — PCA components of {FP_NAME}")
plt.tight_layout()
plt.show()

In [ ]:
# Gini / impurity-based importance (same RF, same order as RDKIT_PHYS_PROP_NAMES)
fig, ax = plt.subplots(figsize=(6, 4.5))
order = np.argsort(rf_phys.feature_importances_)
ax.barh([names[i] for i in order], rf_phys.feature_importances_[order], color="coral")
ax.set_xlabel("Mean decrease in impurity (Gini)")
ax.set_title("Phys RF — feature_importances_")
plt.tight_layout()
plt.show()

In [ ]:
# Which fingerprint bits load most strongly on PC1? (structure-level follow-up: substructure search on those bits)
load1 = pca_fp.components_[0]
topk = np.argsort(np.abs(load1))[-30:][::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(topk)), load1[topk], color="steelblue")
ax.set_xticks(range(len(topk)))
ax.set_xticklabels([str(int(i)) for i in topk], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Bit index (Morgan fingerprint)")
ax.set_ylabel("PC1 loading")
ax.set_title(f"FP PC1 — largest |loading| ({FP_NAME})")
plt.tight_layout()
plt.show()